# Lab 06-06 · Dependent Sources

Companion to chapter [06-06](../../chapters/06-06-dependent-sources.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)

ngspice available: True


## Part 1 — E, G, F and H in ngspice

[dependent.cir](dependent.cir) builds the chapter's examples with SPICE's dependent sources. Current-controlled sources (F, H) need the controlling current to flow through a voltage source, so the netlist inserts 0 V "sense" sources.

**🔮 Predict:** in Example 06-06.3 the sense source `V3s` sits in the base lead, + terminal toward the signal source. What sign will ngspice report for its current?

In [2]:
def run_ngspice(path):
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) == 2:
            try:
                vals[parts[0].strip()] = float(parts[1])
            except ValueError:
                pass
    return vals

expected = {"ex1v2": -1.333333, "ex2gain": -6.211180, "ex3ib": 7.352941e-7, "ex3vo": -0.2941176,
            "ex4a": 1000.0, "ex4b": -1.0e5, "ex5vo": 9.998939}
if HAVE_NGSPICE:
    ng = run_ngspice("dependent.cir")
    for k, x in expected.items():
        print(f"{k:8s} ngspice {ng[k]: .6g}   hand analysis {x: .6g}")
        assert abs(ng[k] - x) <= 2e-5 * abs(x)
else:
    print("ngspice not found; see SETUP.md. Skipping.")

ex1v2    ngspice -1.33333   hand analysis -1.33333
ex2gain  ngspice -6.21118   hand analysis -6.21118
ex3ib    ngspice  7.35294e-07   hand analysis  7.35294e-07
ex3vo    ngspice -0.294118   hand analysis -0.294118
ex4a     ngspice  1000   hand analysis  1000
ex4b     ngspice -100000   hand analysis -100000
ex5vo    ngspice  9.99894   hand analysis  9.99894


The sense current is **positive**, 0.735 µA: SPICE reports current flowing into a source's + terminal, and the base current flows from the signal source into the base, entering `V3s` at its + terminal. For a source that delivers power the same convention gives a negative number (06-01).

## Part 2 — Nonsymmetric matrices

Here is 06-04's mini-SPICE with two more stamps (§3.6 of the chapter): `G name a b c d g` (current $g(v_c - v_d)$ from $a$ through the source to $b$) and `E name a b c d mu` (a voltage source $v_a - v_b = \mu(v_c - v_d)$).

**🔮 Predict:** in Example 06-06.1, which matrix entry has no mirror image?

In [3]:
def mna(netlist):
    elems = [line.split() for line in netlist.strip().splitlines() if line.strip() and not line.startswith("*")]
    nodes = sorted({n for e in elems for n in e[1:3] + (e[3:5] if e[0][0] in "GE" else []) if n != "0"})
    extra = [e[0] for e in elems if e[0][0] in "VE"]
    row = {n: k for k, n in enumerate(nodes)}
    row.update({s: len(nodes) + k for k, s in enumerate(extra)})
    size = len(nodes) + len(extra)
    A, z = np.zeros((size, size)), np.zeros(size)
    def add(r, c, val):
        if r is not None and c is not None:
            A[r, c] += val
    for e in elems:
        kind, a, b = e[0][0], row.get(e[1]), row.get(e[2])
        if kind == "R":
            g = 1 / float(e[3])
            add(a, a, g); add(b, b, g); add(a, b, -g); add(b, a, -g)
        elif kind == "I":
            if a is not None: z[a] -= float(e[3])
            if b is not None: z[b] += float(e[3])
        elif kind == "V":
            m = row[e[0]]
            add(a, m, 1); add(m, a, 1); add(b, m, -1); add(m, b, -1); z[m] = float(e[3])
        elif kind == "G":                       # VCCS: g (v_c - v_d) from a to b
            c, d, g = row.get(e[3]), row.get(e[4]), float(e[5])
            add(a, c, g); add(a, d, -g); add(b, c, -g); add(b, d, g)
        elif kind == "E":                       # VCVS: v_a - v_b - mu (v_c - v_d) = 0
            m, c, d, mu = row[e[0]], row.get(e[3]), row.get(e[4]), float(e[5])
            add(a, m, 1); add(m, a, 1); add(b, m, -1); add(m, b, -1)
            add(m, c, -mu); add(m, d, mu)
    x = np.linalg.solve(A, z)
    return {n: x[row[n]] for n in nodes}, {s: x[row[s]] for s in extra}, A, list(row)

ex1 = '''
I1 0 n1 1e-3
R1 n1 0 2e3
R2 n1 n2 4e3
R3 n2 0 4e3
G1 n2 0 n1 0 1e-3
'''
v, _, A, names = mna(ex1)
print("unknowns:", names)
print("G (mS):\n", np.round(A * 1e3, 4))
print("v1 =", round(v["n1"], 6), "V,  v2 =", round(v["n2"], 6), "V")
print("symmetric?", np.allclose(A, A.T))
assert abs(v["n2"] + 4 / 3) < 1e-9 and not np.allclose(A, A.T)

ex5 = '''
V5 p 0 1
R1 p n 1e6
R2 n 0 1e3
R3 n o 9e3
R4 o 0 2e3
R5 o x 100
E1 x 0 p n 1e5
'''
v5, j5, A5, names5 = mna(ex5)
print("\nExample 06-06.5: v_out =", round(v5["o"], 6), "V;  v+ - v- =", round((v5["p"] - v5["n"]) * 1e6, 3), "uV")
print("largest asymmetry |A - A^T|:", np.abs(A5 - A5.T).max())
assert abs(v5["o"] - 9.998939) < 1e-6

unknowns: ['n1', 'n2']
G (mS):
 [[ 0.75 -0.25]
 [ 0.75  0.5 ]]
v1 = 0.888889 V,  v2 = -1.333333 V
symmetric? False

Example 06-06.5: v_out = 9.998939 V;  v+ - v- = 105.989 uV
largest asymmetry |A - A^T|: 100000.0


In Example 06-06.1 the entry at row $v_2$, column $v_1$ is $-0.25 + 1 = 0.75$ mS, while its mirror is $-0.25$ mS: the VCCS's $+g$ lands only in the driven node's row and the controlling node's column. In Example 06-06.5 the VCVS row holds $-\mu = -10^5$ in one column with nothing to mirror it, the largest asymmetry in the matrix.

## Part 3 — Sweeping the gain: Miller and negative resistance

**🔮 Predict:** for Example 06-06.4's circuit, $R_{in} = R_f/(1 - \mu)$. Where does $R_{in}$ change sign, and what happens at $\mu = 1$?

In [4]:
Rf = 100e3
mu = np.concatenate([np.linspace(-100, 0.9, 400), np.linspace(1.1, 100, 400)])
Rin = Rf / (1 - mu)
# spot-check the formula against the MNA solver at three gains
for m in (-99, 0.5, 2):
    net = f'''
V1 n1 0 1
RF n1 o {Rf}
E1 o 0 n1 0 {m}
'''
    _, j, _, _ = mna(net)
    print(f"mu = {m:5}: R_in from MNA = {-1 / j['V1'] / 1e3: .4f} kOhm, formula = {Rf / (1 - m) / 1e3: .4f} kOhm")
    assert abs(-1 / j["V1"] - Rf / (1 - m)) < 1e-6 * abs(Rf / (1 - m))
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(mu, Rin / 1e3)
ax.axvline(1, color="0.5", ls="--", lw=0.8); ax.axhline(0, color="0.5", lw=0.8)
ax.set_ylim(-250, 250); ax.set_xlabel("VCVS gain μ"); ax.set_ylabel("R_in (kΩ)"); ax.grid(True, alpha=0.3)
plt.show()

mu =   -99: R_in from MNA =  1.0000 kOhm, formula =  1.0000 kOhm
mu =   0.5: R_in from MNA =  200.0000 kOhm, formula =  200.0000 kOhm
mu =     2: R_in from MNA = -100.0000 kOhm, formula = -100.0000 kOhm


$R_{in}$ is positive for $\mu < 1$, infinite at $\mu = 1$ (the output follows the input and no current flows: bootstrapping), and negative for $\mu > 1$. For large inverting gains it falls toward zero: the Miller effect.

## Part 4 — How much loop gain is enough?

**🔮 Predict:** Example 06-06.5's gain error is about $1/(A\beta)$. Over what range of $A$ is the error below 0.1%? How does the error scale with $A$?

In [5]:
def closed_loop(Aol):
    net = ex5.replace("1e5", f"{Aol:g}")
    v, _, _, _ = mna(net)
    return v["o"]

As = np.logspace(1, 7, 25)
err = np.array([(10 - closed_loop(a)) / 10 for a in As])
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.loglog(As, err * 1e6, "o-", label="model")
ax.loglog(As, 1e6 / (As * 0.1), "--", label="1/(Aβ)")
ax.set_xlabel("open-loop gain A"); ax.set_ylabel("gain error (ppm)"); ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.show()
slope = np.polyfit(np.log10(As[-10:]), np.log10(err[-10:]), 1)[0]
print(f"error at A = 1e3: {(10 - closed_loop(1e3)) / 10 * 100:.3f} %;  at A = 1e7: {(10 - closed_loop(1e7)) / 10 * 1e6:.3f} ppm")
print(f"slope at large A: {slope:.3f} decades per decade")
assert abs(slope + 1) < 0.01

error at A = 1e3: 1.050 %;  at A = 1e7: 1.061 ppm
slope at large A: -1.000 decades per decade


The error falls one decade per decade of $A$: halving the error needs twice the gain. It sits slightly above $1/(A\beta)$ because $R_o$ working into the 2 kΩ load and the feedback network costs a little effective gain. At $A = 10^3$ the error is 1.05%; 0.1% needs $A$ of about $10^4$; at $A = 10^7$ it is 1.06 ppm.

## Your turn

1. In Part 2, add a 50 kΩ $r_o$ across Example 06-06.1's VCCS (from node 2 to ground). Predict, then compute, the new $v_2$.
2. In Part 3, what VCVS gain makes $R_{in}$ = −50 kΩ?
3. In Part 4, double $R_o$ to 200 Ω. Does the error at $A = 10^5$ rise or fall, and roughly by how much?

<details><summary>Answers</summary>

1. The 50 kΩ adds 0.02 mS to node 2's diagonal: $0.75v_1 + 0.52v_2 = 0$ with $0.75v_1 - 0.25v_2 = 1$, so $v_2 = -0.75v_1/0.52$, $v_1 = 1/(0.75 + 0.25\times0.75/0.52) = 0.9004$ V and $v_2 = -1.299$ V: slightly less gain ($-1.442$ instead of $-1.5$), because $r_o$ shunts part of the VCCS's current.
2. $R_f/(1 - \mu) = -50$ kΩ gives $1 - \mu = -2$, so $\mu = 3$.
3. It rises: a larger $R_o$ drops more of the VCVS's voltage before the load, so less effective loop gain remains. The loading term roughly doubles, from about 6 ppm to about 12 ppm, so the error goes from 106 ppm to 112 ppm.

</details>